In [1]:
# ============================================================
# Install Required Libraries
# ============================================================

!pip install -q transformers
!pip install -q accelerate
!pip install -q sentencepiece
!pip install -q protobuf

print("=" * 60)
print("Libraries Installed Successfully")
print("=" * 60)

Libraries Installed Successfully


In [2]:
# ============================================================
# Import Libraries
# ============================================================

import torch

from transformers import (

    AutoTokenizer,

    AutoModelForCausalLM

)

print("=" * 60)
print("Libraries Imported Successfully")
print("=" * 60)

Libraries Imported Successfully


In [3]:
# ============================================================
# Check GPU
# ============================================================

device = torch.device(

    "cuda"

    if torch.cuda.is_available()

    else "cpu"

)

print("=" * 60)

print("Device :", device)

if torch.cuda.is_available():

    print("GPU :", torch.cuda.get_device_name(0))

print("=" * 60)

Device : cuda
GPU : NVIDIA GeForce RTX 3050 6GB Laptop GPU


In [4]:
# ============================================================
# Teacher Model
# ============================================================

TEACHER_MODEL = "Qwen/Qwen2.5-3B-Instruct"

teacher_tokenizer = AutoTokenizer.from_pretrained(

    TEACHER_MODEL

)

teacher_model = AutoModelForCausalLM.from_pretrained(

    TEACHER_MODEL,

    torch_dtype=torch.float16,

    device_map="auto"

)

print("=" * 60)
print("Teacher Model Loaded Successfully")
print("=" * 60)

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

C:\Users\sayan\AppData\Local\Programs\Python\Python313\Lib\site-packages\huggingface_hub\file_download.py:139: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\sayan\.cache\huggingface\hub\models--Qwen--Qwen2.5-3B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Some parameters are on the meta device because they were offloaded to the cpu.


Teacher Model Loaded Successfully


In [5]:
# ============================================================
# Freeze Teacher
# ============================================================

for param in teacher_model.parameters():

    param.requires_grad = False

teacher_model.eval()

print("=" * 60)
print("Teacher Model Frozen")
print("=" * 60)

Teacher Model Frozen


In [6]:
# ============================================================
# Verify Teacher
# ============================================================

print("=" * 60)

print("Teacher Model :", TEACHER_MODEL)

print("Device :", next(teacher_model.parameters()).device)

print("Training Mode :", teacher_model.training)

trainable = sum(p.requires_grad for p in teacher_model.parameters())

print("Trainable Parameters :", trainable)

print("=" * 60)

Teacher Model : Qwen/Qwen2.5-3B-Instruct
Device : cuda:0
Training Mode : False
Trainable Parameters : 0


In [7]:
# ============================================================
# Test Teacher Inference
# ============================================================

prompt = "Explain pneumonia in one sentence."

inputs = teacher_tokenizer(

    prompt,

    return_tensors="pt"

).to(device)

with torch.no_grad():

    outputs = teacher_model.generate(

        **inputs,

        max_new_tokens=50,

        do_sample=False

    )

response = teacher_tokenizer.decode(

    outputs[0],

    skip_special_tokens=True

)

print("=" * 60)
print(response)
print("=" * 60)

Explain pneumonia in one sentence. Pneumonia is an inflammatory condition of the lungs, typically caused by infection, that results in the filling of lung tissue with fluid or pus, impairing oxygen exchange and causing symptoms such as cough, fever, and difficulty breathing.
